# Phase 2 - Week 6 - Lesson Day 4
## Model Comparison + Week 6 Mini Challenge

Full explanation is in `Day4 Model Comparison Mini Challenge.pdf`. This is a fully solved mini challenge. After you read it, repeat the same steps on your own dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

---
## Step 1: Create the data and split it

In [ ]:
# Made-up student data: will the student pass (1) or fail (0)?
np.random.seed(42)
n = 300
hours_studied = np.random.uniform(0, 10, n)
attendance_pct = np.random.uniform(50, 100, n)
sleep_hours = np.random.normal(7, 1.2, n)

# The "true" score depends on hours and attendance (sleep does not matter here)
score = hours_studied * 6 + attendance_pct * 0.3 + np.random.normal(0, 8, n)
passed = (score >= 55).astype(int)

df = pd.DataFrame({
    "hours_studied": hours_studied,
    "attendance_pct": attendance_pct,
    "sleep_hours": sleep_hours,
    "passed": passed,
})

X = df[["hours_studied", "attendance_pct", "sleep_hours"]]   # inputs
y = df["passed"]                                              # what we predict

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training rows:', len(X_train), '| Test rows:', len(X_test))

## Step 2: Train three models on the SAME training data

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

## Step 3 and 4: Measure each model on the SAME test data and put it in one table

In [ ]:
rows = []
for name, model in models.items():
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]   # probability of Pass

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

## Step 5: Bar chart of the F1 scores

In [ ]:
results["F1"].plot(kind="bar", color=["#2a6fb0", "#d9731f", "#2f8f5b"], rot=0)
plt.title("F1 Score by Model (test data)")
plt.ylabel("F1 score")
plt.ylim(0, 1)
plt.show()

## Step 6: Conclusion

Look at the table above. In this dataset all three models score close to each other, and Logistic Regression is as good as or better than the tree and the forest. That makes sense: we built the data so passing depends on a fairly straight-line mix of hours studied and attendance, and Logistic Regression is made for that kind of pattern.

**A fair warning:** we only have 60 test students, so the gaps are tiny. Logistic Regression at 0.90 versus 0.867 for the others is a difference of about 2 students. That is not strong proof that one model is truly better. It only tells us they are about equal here.

**My pick:** Logistic Regression. When models score about the same, choose the simplest one. It is faster, easier to explain to other people, and less likely to overfit.

**When I would switch:** if the real pattern were more complicated (for example, "students pass if they study a lot OR attend a lot, but only if they sleep enough"), the tree and the forest could pull ahead, because they can learn that kind of rule.

## Step 7: Predict for a new student

In [ ]:
new_student = pd.DataFrame({
    "hours_studied": [6],
    "attendance_pct": [80],
    "sleep_hours": [7],
})

chosen_model = models["Logistic Regression"]
print("Prediction (1 = pass):", chosen_model.predict(new_student)[0])
print("Probability of fail, pass:", chosen_model.predict_proba(new_student)[0].round(3))

---
## Your Turn

Copy this notebook, load your own dataset, pick a yes/no column to predict, and repeat Steps 1 to 7.